In [5]:
import sys
import os

import pandas as pd
import numpy as np
import pickle
import json
import optuna

project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
if project_root not in sys.path:
    sys.path.append(project_root)

from config.config import BUCKET_NAME, source_loc, output_loc, config_loc

sys.path.insert(0, source_loc)
from utilities.utility_functions import compute_rrf_scores, calculate_recall_at_k, evaluate_ensemble

In [6]:
#Create Empty Dictionary to save best weights
best_weights = {}

##### Read in base model predictions

In [7]:
try:
    lr_preds = pd.read_parquet(os.path.join(output_loc, "lgbm_ranker_oof_preds.parquet"))
    lc_preds = pd.read_parquet(os.path.join(output_loc, "lgbm_classifier_oof_preds.parquet"))
    xr_preds = pd.read_parquet(os.path.join(output_loc, "xgboost_ranker_oof_preds.parquet"))
    xc_preds = pd.read_parquet(os.path.join(output_loc, "xgboost_classifier_oof_preds.parquet"))
    
except FileNotFoundError as e:
    print(f"ERROR: {e}")
    print("Please ensure base model scripts ran successfully and saved outputs to base_models/base_model_preds_params/ ")
    exit(1)

##### Compute Rank Reciprocal Fusion (RRF) Scores

In [8]:
#Create all_preds df
base_model_preds = lr_preds[['user_id', 'anchor_order_number', 'label_reordered', 'lgbm_ranker_pred']].copy()
base_model_preds = base_model_preds.rename(columns= {'lgbm_ranker_pred' : 'lgbm_ranker'})
base_model_preds['lgbm_classifier'] = lc_preds['lgbm_classifier_pred']
base_model_preds['xgb_ranker'] = xr_preds['xgboost_ranker_pred']
base_model_preds['xgb_classifier'] = xc_preds['xgboost_classifier_pred']

#Compute rrf scores for each base model
base_model_preds = compute_rrf_scores(base_model_preds, ['lgbm_ranker', 'lgbm_classifier', 'xgb_ranker', 'xgb_classifier'],
                                      ['user_id', 'anchor_order_number'])

In [9]:
display(base_model_preds.sort_values(by = ['user_id', 'anchor_order_number']))
display(base_model_preds)

,user_id,anchor_order_number,label_reordered,lgbm_ranker,lgbm_classifier,xgb_ranker,xgb_classifier,lgbm_ranker_rrf,lgbm_classifier_rrf,xgb_ranker_rrf,xgb_classifier_rrf
36558,64,5,0,0.095872,0.170823,0.040247,0.177552,0.014286,0.014286,0.014286,0.014286
49385,64,5,0,-0.803198,0.050887,-1.191169,0.051648,0.010526,0.009804,0.010526,0.010417
104798,64,5,0,-0.818351,0.051144,-1.233037,0.051600,0.010417,0.010417,0.010101,0.010309
144156,64,5,0,-0.608366,0.061137,-1.025823,0.060267,0.011905,0.011905,0.011494,0.011364
247021,64,5,0,-0.705128,0.054135,-1.036657,0.059951,0.010989,0.010870,0.011236,0.011111
...,...,...,...,...,...,...,...,...,...,...,...
1336081,206025,8,0,-1.883196,0.031674,-2.079310,0.032781,0.008696,0.008475,0.008696,0.008696
1353394,206025,8,0,-0.275764,0.152395,-0.233038,0.157335,0.013158,0.013889,0.014286,0.013699
1362195,206025,8,0,-1.602375,0.038001,-1.694110,0.040883,0.009615,0.009174,0.009709,0.009434
1387116,206025,8,1,0.914527,0.337035,0.857736,0.340869,0.016393,0.016393,0.016393,0.016393


,user_id,anchor_order_number,label_reordered,lgbm_ranker,lgbm_classifier,xgb_ranker,xgb_classifier,lgbm_ranker_rrf,lgbm_classifier_rrf,xgb_ranker_rrf,xgb_classifier_rrf
0,108133,20,0,-2.446080,0.023082,-2.614035,0.024736,0.006897,0.006536,0.006944,0.006623
1,1310,47,0,-1.305803,0.031139,-0.968362,0.030435,0.007874,0.008197,0.007353,0.008333
2,93022,6,0,-0.620889,0.084619,-0.976385,0.083773,0.012500,0.010753,0.011905,0.010753
3,197603,14,0,-0.451273,0.039348,-0.832354,0.039433,0.013333,0.011765,0.011628,0.011905
4,96653,31,1,-1.273831,0.034070,-1.367580,0.034022,0.010204,0.009804,0.010417,0.010309
...,...,...,...,...,...,...,...,...,...,...,...
1434011,113236,63,0,-2.633651,0.006103,-1.521774,0.005869,0.005102,0.004525,0.004717,0.004717
1434012,205067,24,0,-1.275463,0.069921,-0.689111,0.073565,0.007519,0.008264,0.007937,0.008621
1434013,7610,9,0,-1.098181,0.057991,-1.260865,0.058499,0.009174,0.010101,0.009091,0.010417
1434014,81659,20,0,-0.531295,0.121693,-0.283158,0.118841,0.012658,0.012658,0.012500,0.012658


##### Create optuna objectives and run trials to find optimatal weights for ensemble models

In [10]:
#lgbm ranker + xgboost classifier weight tuning
def obj_ens1(trial):
    w_lr = trial.suggest_float('w_lr', 0.0, 1.0)
    w_xc = trial.suggest_float('w_xc', 0.0, 1.0)

    ensemble_preds = (w_lr/(w_lr+w_xc))*base_model_preds['lgbm_ranker_rrf'] + (w_xc/(w_lr+w_xc))*base_model_preds['xgb_classifier_rrf']
   
    metric_at_5 = evaluate_ensemble(ensemble_preds, base_model_preds)
    
    return metric_at_5

study1 = optuna.create_study(direction='maximize')
study1.optimize(obj_ens1, n_trials=2)
best_weights['ens1_lgb_rank_xgb_class'] = study1.best_params

best_weights

[I 2026-08-02 21:58:51,057] A new study created in memory with name: no-name-5957f2dd-9266-4109-b966-3fd7366f5471
[I 2026-08-02 21:59:06,181] Trial 0 finished with value: 0.5438850445241723 and parameters: {'w_lr': 0.922829093326221, 'w_xc': 0.5824959534610739}. Best is trial 0 with value: 0.5438850445241723.
[I 2026-08-02 21:59:21,195] Trial 1 finished with value: 0.5435838009675162 and parameters: {'w_lr': 0.5552851096652426, 'w_xc': 0.26765654484295875}. Best is trial 0 with value: 0.5438850445241723.


{'ens1_lgb_rank_xgb_class': {'w_lr': 0.922829093326221,
  'w_xc': 0.5824959534610739}}

In [11]:
#lgbm ranker + lgbm classifer weight tuning
def obj_ens2(trial):
    w_lr = trial.suggest_float('w_lr', 0.0, 1.0)
    w_lc = trial.suggest_float('w_lc', 0.0, 1.0)
    
    ensemble_preds = (w_lr/(w_lr+w_lc))*base_model_preds['lgbm_ranker_rrf'] + (w_lc/(w_lr+w_lc))*base_model_preds['lgbm_classifier_rrf']

    metric_at_5 = evaluate_ensemble(ensemble_preds, base_model_preds)
    
    return metric_at_5
    
study2 = optuna.create_study(direction='maximize')
study2.optimize(obj_ens2, n_trials=2)
best_weights['ens2_lgb_rank_lgb_class'] = study2.best_params

best_weights

[I 2026-08-02 21:59:22,881] A new study created in memory with name: no-name-0b4a4db9-c500-4d0b-bd78-97b5650277cb
[I 2026-08-02 21:59:37,923] Trial 0 finished with value: 0.5435673510448693 and parameters: {'w_lr': 0.8577264958549251, 'w_lc': 0.49491995580370096}. Best is trial 0 with value: 0.5435673510448693.
[I 2026-08-02 21:59:53,078] Trial 1 finished with value: 0.5434256482073226 and parameters: {'w_lr': 0.4222913207515322, 'w_lc': 0.36129705124323785}. Best is trial 0 with value: 0.5435673510448693.


{'ens1_lgb_rank_xgb_class': {'w_lr': 0.922829093326221,
  'w_xc': 0.5824959534610739},
 'ens2_lgb_rank_lgb_class': {'w_lr': 0.8577264958549251,
  'w_lc': 0.49491995580370096}}

In [12]:
#xgboost ranker + lgbm classifer weight tuning
def obj_ens3(trial):
    w_xr = trial.suggest_float('w_xr', 0.0, 1.0)
    w_lc = trial.suggest_float('w_lc', 0.0, 1.0)
    
    ensemble_preds = (w_xr/(w_xr+w_lc))*base_model_preds['xgb_ranker_rrf'] + (w_lc/(w_xr+w_lc))*base_model_preds['lgbm_classifier_rrf']

    metric_at_5 = evaluate_ensemble(ensemble_preds, base_model_preds)
    
    return metric_at_5
    
study3 = optuna.create_study(direction='maximize')
study3.optimize(obj_ens3, n_trials=2)
best_weights['ens3_xgb_rank_lgb_class'] = study3.best_params

best_weights

[I 2026-08-02 21:59:53,086] A new study created in memory with name: no-name-86b9b3fa-e889-42fa-b8f6-5357c5246485
[I 2026-08-02 22:00:08,015] Trial 0 finished with value: 0.5419767637040926 and parameters: {'w_xr': 0.08097537954739709, 'w_lc': 0.8993871921338698}. Best is trial 0 with value: 0.5419767637040926.
[I 2026-08-02 22:00:22,892] Trial 1 finished with value: 0.5435564127995318 and parameters: {'w_xr': 0.5669848904121114, 'w_lc': 0.7081500358726892}. Best is trial 1 with value: 0.5435564127995318.


{'ens1_lgb_rank_xgb_class': {'w_lr': 0.922829093326221,
  'w_xc': 0.5824959534610739},
 'ens2_lgb_rank_lgb_class': {'w_lr': 0.8577264958549251,
  'w_lc': 0.49491995580370096},
 'ens3_xgb_rank_lgb_class': {'w_xr': 0.5669848904121114,
  'w_lc': 0.7081500358726892}}

In [13]:
#xgboost ranker + xgboost classifer weight tuning
def obj_ens4(trial):
    w_xr = trial.suggest_float('w_xr', 0.0, 1.0)
    w_xc = trial.suggest_float('w_xc', 0.0, 1.0)
    
    ensemble_preds = (w_xr/(w_xr+w_xc))*base_model_preds['xgb_ranker_rrf'] + (w_xc/(w_xr+w_xc))*base_model_preds['xgb_classifier_rrf']

    metric_at_5 = evaluate_ensemble(ensemble_preds, base_model_preds)
    
    return metric_at_5
    
study4 = optuna.create_study(direction='maximize')
study4.optimize(obj_ens4, n_trials=2)
best_weights['ens4_xgb_rank_xgb_class'] = study4.best_params

best_weights

[I 2026-08-02 22:00:22,899] A new study created in memory with name: no-name-484fa938-ba58-4043-a151-f19019ea4c66
[I 2026-08-02 22:00:37,671] Trial 0 finished with value: 0.5451297337789942 and parameters: {'w_xr': 0.6987537248615024, 'w_xc': 0.18976632409767447}. Best is trial 0 with value: 0.5451297337789942.
[I 2026-08-02 22:00:52,904] Trial 1 finished with value: 0.5449444481907275 and parameters: {'w_xr': 0.7576113116201207, 'w_xc': 0.033600039755308564}. Best is trial 0 with value: 0.5451297337789942.


{'ens1_lgb_rank_xgb_class': {'w_lr': 0.922829093326221,
  'w_xc': 0.5824959534610739},
 'ens2_lgb_rank_lgb_class': {'w_lr': 0.8577264958549251,
  'w_lc': 0.49491995580370096},
 'ens3_xgb_rank_lgb_class': {'w_xr': 0.5669848904121114,
  'w_lc': 0.7081500358726892},
 'ens4_xgb_rank_xgb_class': {'w_xr': 0.6987537248615024,
  'w_xc': 0.18976632409767447}}

In [14]:
#All base model ensemble
def obj_ens5(trial):

    w_lr = trial.suggest_float('w_lr', 0.0, 1.0)
    w_lc = trial.suggest_float('w_lc', 0.0, 1.0)
    w_xr = trial.suggest_float('w_xr', 0.0, 1.0)
    w_xc = trial.suggest_float('w_xc', 0.0, 1.0)
    tot = w_lr + w_lc + w_xr + w_xc
    
    ensemble_preds = ((w_lr/tot)*base_model_preds['lgbm_ranker_rrf'] + (w_lc/tot)*base_model_preds['lgbm_classifier_rrf'] +
                 (w_xr/tot)*base_model_preds['xgb_ranker_rrf'] + (w_xc/tot)*base_model_preds['xgb_classifier_rrf'])

    metric_at_5 = evaluate_ensemble(ensemble_preds, base_model_preds)
    
    return metric_at_5
    
study5 = optuna.create_study(direction='maximize')
study5.optimize(obj_ens5, n_trials=2)
best_weights['ens5_all_base_models'] = study5.best_params

best_weights

[I 2026-08-02 22:00:52,912] A new study created in memory with name: no-name-086cf5dc-097b-4d56-8b03-f2682e2aebb5
[I 2026-08-02 22:01:07,927] Trial 0 finished with value: 0.5449575597459683 and parameters: {'w_lr': 0.32145252775880717, 'w_lc': 0.26575235294748945, 'w_xr': 0.3653203260629626, 'w_xc': 0.29756193378321427}. Best is trial 0 with value: 0.5449575597459683.
[I 2026-08-02 22:01:22,916] Trial 1 finished with value: 0.5450875769007196 and parameters: {'w_lr': 0.2593224615094222, 'w_lc': 0.49264586397633436, 'w_xr': 0.7662361842809281, 'w_xc': 0.14242668156726968}. Best is trial 1 with value: 0.5450875769007196.


{'ens1_lgb_rank_xgb_class': {'w_lr': 0.922829093326221,
  'w_xc': 0.5824959534610739},
 'ens2_lgb_rank_lgb_class': {'w_lr': 0.8577264958549251,
  'w_lc': 0.49491995580370096},
 'ens3_xgb_rank_lgb_class': {'w_xr': 0.5669848904121114,
  'w_lc': 0.7081500358726892},
 'ens4_xgb_rank_xgb_class': {'w_xr': 0.6987537248615024,
  'w_xc': 0.18976632409767447},
 'ens5_all_base_models': {'w_lr': 0.2593224615094222,
  'w_lc': 0.49264586397633436,
  'w_xr': 0.7662361842809281,
  'w_xc': 0.14242668156726968}}

##### Save optimal ensemble weights locally

In [17]:
best_weights_path = os.path.join(config_loc, "optimal_ensemble_weights.json")
with open(best_weights_path, "w") as f:
    json.dump(best_weights, f, indent=4)

In [16]:
best_weights

{'ens1_lgb_rank_xgb_class': {'w_lr': 0.922829093326221,
  'w_xc': 0.5824959534610739},
 'ens2_lgb_rank_lgb_class': {'w_lr': 0.8577264958549251,
  'w_lc': 0.49491995580370096},
 'ens3_xgb_rank_lgb_class': {'w_xr': 0.5669848904121114,
  'w_lc': 0.7081500358726892},
 'ens4_xgb_rank_xgb_class': {'w_xr': 0.6987537248615024,
  'w_xc': 0.18976632409767447},
 'ens5_all_base_models': {'w_lr': 0.2593224615094222,
  'w_lc': 0.49264586397633436,
  'w_xr': 0.7662361842809281,
  'w_xc': 0.14242668156726968}}